In [ ]:
# -------------------------------
# Imports and setup
# -------------------------------
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Append local repo folder
sys.path.append("/path/to/your/CausalICM/")  # <-- update this path
print(sys.path)

from functions.data_simulation import data_simulation8
from functions.model_training import ICM
from functions.weightedMSE import weighted_mean_squared_error

# -------------------------------
# Set seeds for reproducibility
# -------------------------------
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)

# -------------------------------
# Generate data
# -------------------------------
data_exp, data_obs = data_simulation8(1000, beta0=-3.0, beta1=-3.0)
data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Logistic regression for trial participation
# -------------------------------
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]
data_full = np.c_[data_full, 1/prob_part]


# -------------------------------
# Rho tuning via cross-validation
# -------------------------------
from sklearn.model_selection import KFold

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5
min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True)
for rho in rho_values:
    avg_wmse = 0
    wmse_per_fold = []
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1]
        X_val, T_val, Y_val, w = val_data[:,1], val_data[:,2], val_data[:,3], val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1],
            X_O=train_data[train_data[:,0]==0,1],
            T_E=train_data[train_data[:,0]==1,2],
            T_O=train_data[train_data[:,0]==0,2],
            Y_E=train_data[train_data[:,0]==1,3],
            Y_O=train_data[train_data[:,0]==0,3],
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

    avg_wmse /= num_folds
    wmse_per_rho[rho] = wmse_per_fold
    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")
pd.DataFrame([best_rho], columns=['best_rho']).to_csv('sim8_best_rho.csv', index=False)

# -------------------------------
# Train ICM with best rho and evaluate
# -------------------------------
best_rho = pd.read_csv('sim8_best_rho.csv').values[0][0]

m0, m1 = ICM(X_E=data_exp[:,1], X_O=data_obs[:,1],
             T_E=data_exp[:,2], T_O=data_obs[:,2],
             Y_E=data_exp[:,3], Y_O=data_obs[:,3],
             r=2, ID=1, AD=0, rho=best_rho)

test_data = np.arange(np.min(np.r_[data_exp[:,1], data_obs[:,1]]),
                      np.max(np.r_[data_exp[:,1], data_obs[:,1]]), 0.01)

predY0_exp = m0.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape)])
predY1_exp = m1.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape)])
predCATE_exp = predY1_exp[0] - predY0_exp[0]

true_cate = 1 + test_data + test_data**2
rmse_exp = mean_squared_error(predCATE_exp, true_cate, squared=False)
print("RMSE - ICMexp:", rmse_exp)


